# 第17章 多轮优化实战

**操作手册** | 运行 Agent、读取轨迹、失败回退与对比报告

本手册对应文档：`docs/part3-agent/chapter17/index.md`  
本手册对应代码：`code/part3-agent/`

---


## 在云端运行本章

正文中的 0.705 ms → 0.322 ms（约 2.19×）来自 Radeon 8060S（`gfx1151`）历史运行。本章会在当前云端 GPU 上重新运行完整闭环，结果以新工作区中的 `trajectory.jsonl` 为准。

模型接口沿用第 16 章配置。若尚未获取 API Key，请先参考 [AMD Radeon Cloud 使用教程](../../docs/cloud/amd-radeon-cloud/index.md) 进入平台，再按第 16 章的 Token Factory 步骤获取。


## 本章目标、前置知识与产物

本章是算子 Agent 篇的高潮：把第 15 章的工具与第 16 章的循环放在一起，跑一次完整多轮优化，然后——**如实展示结果**。

本章主角是 **`vector_add` fixtures**，硬件为 **Radeon 8060S（`gfx1151`）+ ROCm 7.12**，工作区 `task-20260806-144647`。

先说结论，这也是本书和「Agent 自动优化一切」类教程的区别：

- 在**故意留头寸**的朴素 `vector_add` 上，本轮 Agent 搜索把延迟从约 **0.705 ms → 0.322 ms（≈2.19×）**，10 轮评估中 **5 次接受**；
- 加速主要来自 **更大 `block_size` + 更高 `num_warps`**（减 grid 启动、提高并行度），符合 memory-bound 直觉；
- 教程承诺的「3–5×」针对**有明显优化空间**的算子（融合 softmax、naive reduction/matmul 等）。`vector_add` 靠近带宽墙，用来证明**闭环可信**与**失败回退**，不是证明极限加速。

学完本章，你应该能够：

- 按留痕规则读懂 `trajectory.jsonl`；
- 从性能曲线指出哪几轮提升最大、哪几轮该回退；
- 写一份区分事实与假设的对比报告。

对应代码与产物：

```text
code/part3-agent/chapter15/
├── fixtures/vector_add/
│   ├── baseline.py
│   ├── reference.py
│   └── task.json
└── logs/tasks/task-20260806-144647/   # 本章引用的真实工作区
    ├── trajectory.jsonl
    ├── best.py
    ├── hardware.json
    └── viz/

code/part3-agent/chapter16/
├── run_part3_test.py
├── visualize_trajectory.py
└── run_and_visualize.sh
```



## 17.1 选一个教学算子

| 算子 | 用途 | 预期 |
|---|---|---|
| **vector_add**（本章默认） | 验证 Agent 闭环、评测器、非交互入口 | 中小幅到约 2×（视 baseline 有多朴素）；重在轨迹完整 |
| masked-softmax / reduction / naive matmul | 冲击 3–5× 叙事 | 融合、LDS、向量化等机制有头寸 |

选型原则：

1. 有可执行的 Triton baseline 与 PyTorch reference；
2. `costModel` 能区分 bound 类型；
3. 故意留一点「可改空间」（如偏小 `block_size`），否则 Agent 只能在噪声里打转。

本章任务契约摘要：

| 字段 | 内容 |
|---|---|
| Objective | 元素级向量加，与 reference 一致 |
| Shape / dtype | `4096×2048`，fp16，约 8.4M 元素 |
| Correctness | 与 PyTorch reference 比对 |
| Promotion | 配对中位改进 ≥ 1%，多数配对为正 |
| Constraints | Triton on ROCm |



## 17.2 记录每轮优化的轨迹

优化之前先定「留痕规则」。工作区固定若干证据文件：

```text
task-20260806-144647/
├── task.json / reference.py / baseline.py / best.py
├── hardware.json          # measure_peak 副本
├── trajectory.jsonl       # 每行一轮：change / status / accepted / latencyMs / …
├── run_summary.json
└── viz/                   # 可视化 PNG
```

规则只有一条：**任何候选版本，无论成败，都留下记录。**

本轮 Agent 步内工具序列（历史跑次工具名曾为 `evaluate_candidate` / `profile`；**当前代码**请用三件套 + `accept_candidate`）：

| Agent 步 | 当时工具 | 现今对应 | 关键返回 |
|---|---|---|---|
| 1 | `get_environment` | 同左 | Radeon 8060S / gfx1151 / ROCm 7.12 |
| 1 | `measure_peak` | 同左 | 带宽 **210.5 GB/s**；fp16 **27.8 TFLOPS** |
| 2 | `profile` | `profile_kernel` | AI≈**0.167** → **memory-bound** |
| 3–12 | `evaluate_candidate` ×10 | `compile`+`bench`+`accept` | **5 接受 / 5 拒绝** |



## 17.3 观察性能提升曲线

硬件与算子画像（优化前）：

| 指标 | 数值 |
|---|---|
| 设备 | Radeon 8060S Graphics · `gfx1151` |
| ROCm / torch | 7.12 / `2.10.0+rocm7.12.0` |
| 带宽峰值 | 210.5 GB/s |
| AI / bound | 0.167 · memory-bound |
| Baseline 特征 | `block_size=256`，未设 `num_warps` |
| Baseline 中位延迟（反推） | ≈ **0.705 ms** |

逐轮账本（`improvementFraction` 相对**当时 incumbent**，不是相对最初 baseline）：

| 轮 | 改动 | 延迟 (ms) | 相对 incumbent | 裁决 |
|---|---|---|---|---|
| 1 | `block_size` 256→1024 | 0.378 | **+46.34%** | ✓ 接受 |
| 2 | + `num_stages=2` | 0.372 | -0.32% | ✗ 阈值下 |
| 3 | `block_size`→2048 | 0.342 | -1.01% | ✗ |
| 4 | `block_size`→4096 | 0.338 | **+4.64%** | ✓ 接受 |
| 5 | `block_size`→8192 | 0.405 | -17.34% | ✗ 明显变慢 |
| 6 | 4096 + `num_stages=2` | 0.340 | +0.34% | ✗ &lt;1% |
| 7 | 4096 + `num_warps=8` | 0.329 | **+3.46%** | ✓ 接受 |
| 8 | `num_warps=16` | 0.326 | **+1.24%** | ✓ 接受 |
| 9 | `num_warps=32` | **0.322** | **+1.50%** | ✓ 接受（最终 best） |
| 10 | + `num_stages=2` | 0.323 | +0.50% | ✗ &lt;1% |

相对最初 baseline：

| 指标 | Baseline | Best（R9） | 变化 |
|---|---|---|---|
| 中位延迟 | ≈ 0.705 ms | **0.322 ms** | **≈2.19×**（延迟降约 54%） |
| `block_size` | 256 | **4096** | grid 启动次数约减 16× |
| `num_warps` | 默认 | **32** | 提高 CU 占用与并行度 |

曲线形状：

- **最大单轮收益**来自 R1 放大 `block_size`（配置搜索，减启动开销）；
- R7–R9 的 `num_warps` 阶梯是二次提升；
- R5 `block_size=8192` 是结构/资源边界上的负结果——过大 block 伤害占用或调度。

### 可视化

#### 延迟与配对改进总览

![延迟曲线与配对改进百分比](../../docs/part3-agent/chapter17/images/rounds_overview.png)

上图绿线为 best-so-far latency；下图红虚线为 1% 接受阈值；绿柱接受、橙柱拒绝。R5 大负柱对应 8192 失败试探。

#### 每轮改动时间线

![每轮改动与裁决时间线](../../docs/part3-agent/chapter17/images/process_timeline.png)

#### 状态分布

![接受与拒绝状态分布](../../docs/part3-agent/chapter17/images/status_breakdown.png)

10 轮：**接受 = 5**，**未达阈值 = 5**，本轮无编译失败。

最终 `best.py` 要点：

```python
block_size = 4096
_vadd_kernel[grid](..., block_size, num_warps=32)
```



## 17.4 失败回退机制

回退不是单独的「undo 工具」，而是评测器契约：

```text
候选失败或未过阈值 → accepted=false → 不写 best.py → incumbent 保持上一版
```

本轮最有教学价值的失败是 **R5：`block_size=8192`**——编译与正确性都过，但中位改进 **-17.3%**，明显变慢。Agent 拒绝后继续从 4096 路线搜索 `num_warps`，没有把坏候选写进 best。

失败记录的价值：

> 「过大 block 在本机 vector_add 上是负优化」这条结论，来自一次失败实验，却能阻止后续盲目把 `block_size` 推到极限。

Agent 侧配合：把拒绝原因读进下一轮提示；SOP 要求「连续同类失败就换机制」；轨迹保留失败轮次，便于报告诚实引用。



## 17.5 和人工优化的对比

| 维度 | 本轮 Agent | 人工典型做法 |
|---|---|---|
| 发现方向 | 在 memory-bound SOP 引导下试 block / warps | 一次选定较大 block + 合理 warps |
| 执行与留痕 | 10 轮全自动评测 + `trajectory.jsonl` | 常靠笔记，易丢负结果 |
| 最终结果 | ≈2.19× vs 故意朴素 baseline | 熟练者可能更少轮次到达相近点 |
| 局限 | 未发明新算法结构；步数耗尽即停 | 需要人盯着跑实验 |

诚实结论：

1. Agent 当前强项是 **执行 + 记录**：给定方向空间，能快速产出变体并自动验证；
2. Agent 弱项仍是 **发现全新结构**（线上 fused MLP 对照实验里纯 Agent 贡献接近 0%——见线上第 17.5 节）；
3. 「3–5×」的正确打开方式往往是 **人给关键洞察，Agent 快速验证并留痕**。本轮 2.19× 发生在「baseline 故意很差」的前提下，不要外推到已经接近带宽墙的生产 kernel。



## 17.6 生成对比报告

好的报告回答五个问题：

| 问题 | 本轮回答 |
|---|---|
| 题目是什么 | `vector_add` fp16 `4096×2048`；配对改进 ≥1% |
| 每轮改了什么 | 上表：block_size / num_warps / num_stages |
| 哪些失败了、为什么 | R5 过大 block；多次 `num_stages` 未过阈值 |
| 最终结论 | 0.322 ms，≈2.19×；best = 4096 + num_warps=32 |
| 事实 vs 推测 | 轨迹与 `hardware.json` 是事实；「还能再快」是推测 |

更细的工具调用链与指标说明见同目录 [optimization-report.md](../../docs/part3-agent/chapter17/optimization-report.md)。

### 本地复现（可选）

下面的命令面向自行配置的 AMD + ROCm 本地环境，**不是运行本 Notebook 的必做步骤**。本地读者可按原文入口准备环境并运行一键脚本：

```bash
cd code/part3-agent
uv sync && source ./activate-rocm.sh

# 需要 ~/.config/hello-gpu/kernel-agent.env（勿提交 git）
bash chapter16/run_and_visualize.sh --skip-pytest
```

本地环境可先执行以下检查：

```bash
uv run python -c "import torch; print(torch.cuda.get_device_name(0), torch.__version__)"
# 本机实测示例：Radeon 8060S Graphics  2.10.0+rocm7.12.0
```

云端已预装 ROCm、PyTorch 和 Triton，可以跳过 `uv sync` 与 `source ./activate-rocm.sh`。直接继续下方 **Execution**；Notebook 会使用当前 Jupyter kernel，并检查或补充 Agent 所需的 Python 第三方包。



## Execution

### 步骤1：定位仓库根目录


In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import re
import subprocess
import sys


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        for repo in (candidate, candidate / "hello-gpu"):
            if (
                (repo / "code/part3-agent/kernel_optimize/agent.py").is_file()
                and (repo / "notebooks/part3-agent").is_dir()
            ):
                return repo
    raise FileNotFoundError("未找到 hello-gpu 仓库；请从仓库目录或其父目录打开本 Notebook。")


REPO_ROOT = find_repo_root()
PART3_ROOT = REPO_ROOT / "code/part3-agent"
FIXTURE_ROOT = PART3_ROOT / "chapter15/fixtures/vector_add"
CHAPTER16_CODE = PART3_ROOT / "chapter16"
for source_path in (PART3_ROOT, CHAPTER16_CODE):
    if str(source_path) not in sys.path:
        sys.path.insert(0, str(source_path))

print(f"仓库根目录: {REPO_ROOT}")
print(f"代码目录: {PART3_ROOT}")
print(f"Python: {sys.executable}")


### 步骤2：准备依赖、中文字体与模型接口

PyTorch、Triton 和 ROCm 使用云端已有环境。若当前环境没有中文字体，本单元会通过 jsDelivr CDN 下载 Noto Sans CJK SC，不访问 GitHub Raw。也可以设置 `HELLO_GPU_CJK_FONT_URL` 使用平台内部地址，或手工上传字体到 `~/.local/share/fonts/NotoSansCJKsc-Regular.otf`。字体会注册给当前 Matplotlib 进程，避免可视化中的中文缺字警告。

模型默认使用 Radeon Cloud。若要改用 DeepSeek 官方 API，请在本单元末尾注释 Radeon Cloud 四行配置，再取消 DeepSeek 官方四行配置的注释。官方参数参考 [首次调用 API](https://api-docs.deepseek.com/zh-cn/)；Key 在 [DeepSeek 开放平台](https://platform.deepseek.com/api_keys) 创建。切换平台时会重新要求粘贴对应 Key。


In [ ]:
requirements = {
    "litellm": "litellm>=1.50",
    "pydantic": "pydantic>=2.0",
    "prompt_toolkit": "prompt-toolkit>=3.0",
    "fastapi": "fastapi>=0.100",
    "orjson": "orjson>=3.0",
    "matplotlib": "matplotlib>=3.0",
}
missing = [package for module, package in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    print("安装缺失依赖:", " ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
    importlib.invalidate_caches()
else:
    print("Python 依赖已就绪")

from matplotlib import font_manager, rcParams
from matplotlib.ft2font import FT2Font
import shutil
import urllib.request

CJK_FONT_CANDIDATES = (
    "Noto Sans CJK SC",
    "Source Han Sans SC",
    "WenQuanYi Micro Hei",
    "WenQuanYi Zen Hei",
)
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
cjk_font_name = next(
    (name for name in CJK_FONT_CANDIDATES if name in available_fonts),
    None,
)

if cjk_font_name is None:
    font_dir = Path.home() / ".local/share/fonts"
    font_path = font_dir / "NotoSansCJKsc-Regular.otf"
    font_relative_path = (
        "notofonts/noto-cjk@main/Sans/OTF/"
        "SimplifiedChinese/NotoSansCJKsc-Regular.otf"
    )
    configured_font_url = os.environ.get("HELLO_GPU_CJK_FONT_URL", "").strip()
    font_urls = [
        url
        for url in (
            configured_font_url,
            f"https://cdn.jsdelivr.net/gh/{font_relative_path}",
            f"https://gcore.jsdelivr.net/gh/{font_relative_path}",
        )
        if url
    ]
    font_dir.mkdir(parents=True, exist_ok=True)
    if not font_path.is_file():
        temporary_font = font_path.with_suffix(".download")
        download_errors = []
        for font_url in font_urls:
            temporary_font.unlink(missing_ok=True)
            print(f"尝试下载中文字体: {font_url}")
            try:
                with urllib.request.urlopen(font_url, timeout=120) as response:
                    with temporary_font.open("wb") as output:
                        shutil.copyfileobj(response, output)
                if temporary_font.stat().st_size < 1_000_000:
                    raise RuntimeError("下载的字体文件异常小")
                temporary_font.replace(font_path)
                print(f"中文字体下载完成: {font_path}")
                break
            except Exception as error:
                download_errors.append(f"{font_url}: {type(error).__name__}: {error}")
            finally:
                temporary_font.unlink(missing_ok=True)
        if not font_path.is_file():
            raise RuntimeError(
                "中文字体下载失败。可将 NotoSansCJKsc-Regular.otf 手工上传到 "
                f"{font_path}，或设置 HELLO_GPU_CJK_FONT_URL。\n"
                + "\n".join(download_errors)
            )

    font_manager.fontManager.addfont(str(font_path))
    cjk_font_name = font_manager.FontProperties(fname=str(font_path)).get_name()
    if shutil.which("fc-cache"):
        subprocess.run(
            ["fc-cache", "-f", str(font_dir)],
            capture_output=True,
            text=True,
            check=False,
        )

font_file = font_manager.findfont(
    font_manager.FontProperties(family=cjk_font_name),
    fallback_to_default=False,
)
required_chars = "接受未达阈值编译失败拒绝未知占比次数计数状态分布"
font_charmap = FT2Font(font_file).get_charmap()
missing_chars = [char for char in required_chars if ord(char) not in font_charmap]
if missing_chars:
    raise RuntimeError(f"中文字体缺少字符: {''.join(missing_chars)}")

os.environ["HELLO_GPU_CJK_FONT"] = str(font_file)
rcParams["font.family"] = [cjk_font_name]
rcParams["font.sans-serif"] = [cjk_font_name, "DejaVu Sans"]
rcParams["axes.unicode_minus"] = False
print(f"Matplotlib 中文字体: {cjk_font_name} ({font_file})")

from getpass import getpass

# Radeon Cloud（默认）：保留下面四行。
PROVIDER_LABEL = "Radeon Cloud"
API_BASE = "https://developer.amd.com.cn/radeon/api/v1"
MODEL = "openai/DeepSeek-V4-Flash"
EXTRA_BODY = {"enable_thinking": False}

# DeepSeek 官方：注释上面四行，再取消下面四行的注释。
# PROVIDER_LABEL = "DeepSeek 官方"
# API_BASE = "https://api.deepseek.com"
# MODEL = "deepseek/deepseek-v4-flash"
# EXTRA_BODY = {"thinking": {"type": "disabled"}}

credential_profile = f"{MODEL}|{API_BASE}"
api_key = os.environ.get("KERNEL_AGENT_API_KEY", "").strip()
if (
    not api_key
    or os.environ.get("KERNEL_AGENT_CREDENTIAL_PROFILE") != credential_profile
):
    api_key = getpass(f"粘贴 {PROVIDER_LABEL} API Key（输入不会显示）: ").strip()
    if not api_key:
        raise ValueError("API Key 不能为空")

os.environ["KERNEL_AGENT_MODEL"] = MODEL
os.environ["KERNEL_AGENT_API_BASE"] = API_BASE
os.environ["KERNEL_AGENT_API_KEY"] = api_key
os.environ["KERNEL_AGENT_EXTRA_BODY"] = json.dumps(EXTRA_BODY)
os.environ["KERNEL_AGENT_CREDENTIAL_PROFILE"] = credential_profile
os.environ["OPENAI_API_KEY"] = api_key
if MODEL.startswith("deepseek/"):
    os.environ["DEEPSEEK_API_KEY"] = api_key

print(f"Provider: {PROVIDER_LABEL}")
print(f"Model: {MODEL}")
print(f"API Base: {API_BASE}")
print(f"Extra Body: {json.dumps(EXTRA_BODY, ensure_ascii=False)}")
print("API Key: configured")


### 步骤3：检查模型与 GPU 环境

平台模型接口限制为 **20 RPM**。本单元会把所有模型请求串行化，并保证两次请求至少间隔 3 秒；公共模型返回 429 时，只对限流错误按 `5 / 10` 秒退避重试。


In [ ]:
import threading
import time
import torch
import triton
from kernel_optimize import llm

MODEL_RPM_LIMIT = 20
MIN_MODEL_REQUEST_INTERVAL_S = 60.0 / MODEL_RPM_LIMIT
RATE_LIMIT_BACKOFF_S = (5, 10)

if not hasattr(llm, "_hello_gpu_original_chat"):
    llm._hello_gpu_original_chat = llm.chat
_original_llm_chat = llm._hello_gpu_original_chat
_model_request_lock = threading.Lock()
_last_model_request_started = 0.0


def is_rate_limit_error(error):
    text = str(error).lower()
    return (
        getattr(error, "status_code", None) == 429
        or error.__class__.__name__ == "RateLimitError"
        or ("429" in text and ("rate limit" in text or "rate_limit" in text))
    )


def retry_after_seconds(error):
    headers = getattr(error, "litellm_response_headers", None)
    if not headers:
        response = getattr(error, "response", None)
        headers = getattr(response, "headers", None) if response is not None else None
    if not headers:
        return None
    value = headers.get("retry-after") or headers.get("Retry-After")
    try:
        return max(0.0, float(value))
    except (TypeError, ValueError):
        return None


def rate_limited_chat(messages, **kwargs):
    global _last_model_request_started
    with _model_request_lock:
        for attempt in range(len(RATE_LIMIT_BACKOFF_S) + 1):
            elapsed = time.monotonic() - _last_model_request_started
            pacing_wait = MIN_MODEL_REQUEST_INTERVAL_S - elapsed
            if pacing_wait > 0:
                time.sleep(pacing_wait)

            _last_model_request_started = time.monotonic()
            try:
                return _original_llm_chat(messages, **kwargs)
            except Exception as error:
                if not is_rate_limit_error(error):
                    raise
                if attempt >= len(RATE_LIMIT_BACKOFF_S):
                    raise
                server_wait = retry_after_seconds(error)
                wait_seconds = max(
                    MIN_MODEL_REQUEST_INTERVAL_S,
                    server_wait if server_wait is not None else RATE_LIMIT_BACKOFF_S[attempt],
                )
                print(
                    f"模型接口 429 限流，{wait_seconds:.0f} 秒后重试 "
                    f"({attempt + 1}/{len(RATE_LIMIT_BACKOFF_S)})"
                )
                time.sleep(wait_seconds)


llm.chat = rate_limited_chat
MODEL_API_READY = False
try:
    message = llm.chat(
        [{"role": "user", "content": "请只回复 READY"}],
        temperature=0.0,
    )
except Exception as error:
    if not is_rate_limit_error(error):
        raise
    print("公共模型持续限流。请稍后重新运行本单元，再继续步骤5。")
else:
    MODEL_API_READY = True
    print(f"Model response: {message.content}")
    print(
        f"请求节流已启用：RPM={MODEL_RPM_LIMIT}，"
        f"最小间隔={MIN_MODEL_REQUEST_INTERVAL_S:.1f}s"
    )

if not torch.cuda.is_available():
    raise RuntimeError("当前 Python 未检测到 ROCm GPU")
properties = torch.cuda.get_device_properties(0)
GPU_ARCH = str(getattr(properties, "gcnArchName", "")).split(":", 1)[0]
GPU_NAME = torch.cuda.get_device_name(0)
print(f"GPU: {GPU_NAME}")
print(f"GPU arch: {GPU_ARCH or 'unknown'}")
print(f"ROCm: {torch.version.hip}")
print(f"PyTorch: {torch.__version__}")
print(f"Triton: {triton.__version__}")


### 步骤4：建立本次实战工作区并测 baseline

工作区使用 `vector_add` fixture。正式搜索前先通过 `bench_kernel` 记录当前设备上的 baseline，后续报告不反推历史数字。


In [ ]:
from datetime import datetime, timezone
from kernel_optimize.__main__ import _seed_workspace_from_fixture
from kernel_optimize.tools import Workspace, build_tools

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
WORKSPACE_DIR = REPO_ROOT / "runs/part3-agent/chapter17" / run_id
_seed_workspace_from_fixture(FIXTURE_ROOT, WORKSPACE_DIR)

workspace = Workspace(WORKSPACE_DIR)
executor, _ = build_tools(workspace, batch=True)
baseline_source = workspace.best_path.read_text(encoding="utf-8")
BASELINE_BENCH = json.loads(executor.call("bench_kernel", {"source": baseline_source}))
(WORKSPACE_DIR / "baseline-benchmark.json").write_text(
    json.dumps(BASELINE_BENCH, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print(f"工作区: {WORKSPACE_DIR}")
print("Baseline benchmark:")
print(json.dumps(BASELINE_BENCH, ensure_ascii=False, indent=2))
if not BASELINE_BENCH.get("ok"):
    raise RuntimeError("baseline 未通过评测，停止多轮优化")


### 步骤5：运行完整多轮优化

模型负责提出候选；compile、bench、profile 和 accept 仍由第 15 章的权威工具完成。每次候选无论接受或拒绝都会写入轨迹。

本步骤沿用步骤3的 20 RPM 节流与 429 退避。重试耗尽时保留当前工作区；等待后重新运行本步骤即可从当前 `best.py` 继续。
当前拆分工具模式使用 25 个模型步骤。batch 主循环不会向模型暴露 `run_code`；候选通过 `bench_kernel` 后，必须先用同一份 source 完成 `accept_candidate` 裁决，才能开始下一个候选。达到最大步数时，最后一个已完成 benchmark 的候选会自动提交权威裁决。
若步数耗尽时最后一个候选已经成功 benchmark，主循环会自动调用 `accept_candidate`；若它只有 compile 结果或 benchmark 未通过，则该未完成候选不计入轨迹，前面已经完成裁决的轨迹仍可继续分析，并在 `agent-status.json` 中标记为 `complete_with_warning`。


In [ ]:
from kernel_optimize.agent import run_agent

MAX_STEPS = 25


def print_step(step, action, observation):
    if action.startswith("tool:"):
        print(f"[step {step}] {action.split(':', 1)[1]}")
    elif action.startswith("result:") and observation:
        first_line = observation.strip().splitlines()[0] if observation.strip() else ""
        print(f"           {first_line[:200]}")
    elif action == "nudge":
        print(f"[step {step}] continue")
    elif action == "done":
        print(f"[step {step}] done")


AGENT_REPORT = None
AGENT_COMPLETED = False
if not MODEL_API_READY:
    print("模型接口检查尚未通过；请先重新运行步骤3。")
else:
    try:
        AGENT_REPORT = run_agent(
            WORKSPACE_DIR,
            max_steps=MAX_STEPS,
            on_step=print_step,
            batch=True,
        )
    except Exception as error:
        if not is_rate_limit_error(error):
            raise
        print("公共模型在重试后仍返回 429，本次 Agent 暂停。")
        print(f"工作区已保留: {WORKSPACE_DIR}")
        print("请等待平台限流恢复后重新运行步骤5，再继续后续单元。")
    else:
        status_path = WORKSPACE_DIR / "agent-status.json"
        agent_status = (
            json.loads(status_path.read_text(encoding="utf-8"))
            if status_path.is_file()
            else {}
        )
        AGENT_COMPLETED = agent_status.get("evidenceReady") is True
        if agent_status.get("state") == "complete_with_warning":
            print("最后一个候选未完成，已忽略该候选；已有权威轨迹仍可分析。")
        if not AGENT_COMPLETED:
            print("Agent 未完成 accept_candidate 裁决，未生成可分析轨迹。")
        (WORKSPACE_DIR / "agent-report.md").write_text(
            AGENT_REPORT + "\n", encoding="utf-8"
        )
        print("\n═══ Agent Report ═══")
        print(AGENT_REPORT)


### 步骤6：读取轨迹并复测最终 best

搜索结束后重新对 `best.py` 调用 `bench_kernel`。表中的改进比例仍表示候选相对当时 incumbent 的配对结果。


In [ ]:
from visualize_trajectory import load_trajectory, print_trajectory_table

threshold = float(workspace.task()["optimization"]["minImprovementFraction"])
TRAJECTORY = []
best_source = workspace.best_path.read_text(encoding="utf-8")
BEST_BENCH = {}
BEST_PROFILE = {}

if not AGENT_COMPLETED:
    print("步骤5尚未完成：请等待限流恢复并重新运行步骤5，本单元未复测 best。")
else:
    TRAJECTORY = load_trajectory(WORKSPACE_DIR / "trajectory.jsonl")
    print_trajectory_table(TRAJECTORY, threshold=threshold)

    BEST_BENCH = json.loads(executor.call("bench_kernel", {"source": best_source}))
    BEST_PROFILE = json.loads(executor.call("profile_kernel", {"source": best_source}))
    (WORKSPACE_DIR / "best-benchmark.json").write_text(
        json.dumps(BEST_BENCH, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    (WORKSPACE_DIR / "best-profile.json").write_text(
        json.dumps(BEST_PROFILE, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

    print("\nBest benchmark:")
    print(json.dumps(BEST_BENCH, ensure_ascii=False, indent=2))
    print("\nBest profile:")
    print(json.dumps(BEST_PROFILE, ensure_ascii=False, indent=2))


### 步骤7：生成当前运行的可视化

三张图写入本次工作区的 `viz/`：轮次总览、状态分布和改动时间线。


In [ ]:
VISUALIZATION_PATHS = []
if not AGENT_COMPLETED:
    print("步骤5尚未完成：未生成当前运行的可视化。")
else:
    from IPython.display import Image, display
    from matplotlib import font_manager as mpl_font_manager
    from matplotlib import rcParams as mpl_rc_params
    import visualize_trajectory

    configured_font = Path(os.environ.get("HELLO_GPU_CJK_FONT", "")).expanduser()
    if not configured_font.is_file():
        raise RuntimeError("未找到已配置的中文字体；请先重新运行步骤2。")

    def force_notebook_cjk_font():
        # addfont 不持久化，因此每次绘图都用 Notebook 下载的绝对路径重新注册。
        mpl_font_manager.fontManager.addfont(str(configured_font))
        family = mpl_font_manager.FontProperties(
            fname=str(configured_font)
        ).get_name()
        mpl_rc_params["font.family"] = [family]
        mpl_rc_params["font.sans-serif"] = [family, "DejaVu Sans"]
        mpl_rc_params["axes.unicode_minus"] = False

    visualize_trajectory._setup_cjk_font = force_notebook_cjk_font
    force_notebook_cjk_font()
    VISUALIZATION_PATHS = visualize_trajectory.render_visualizations(
        WORKSPACE_DIR,
        out_dir=WORKSPACE_DIR / "viz",
        threshold=threshold,
        title=f"part3-agent · {GPU_NAME} · {run_id}",
    )
    for image_path in VISUALIZATION_PATHS:
        print(image_path)
        if image_path.suffix.lower() == ".png":
            display(Image(filename=str(image_path)))


### 步骤8：生成对比报告

报告只使用当前工作区中的 benchmark、profile 和 trajectory。历史 2.19× 仍保留在正文中作为参考，不写入当前结论。


In [ ]:
SUMMARY = None
if not AGENT_COMPLETED:
    print("步骤5尚未完成：未生成当前运行的对比报告。")
else:
    baseline_ms = BASELINE_BENCH.get("median_ms")
    best_ms = BEST_BENCH.get("median_ms")
    speedup = (
        baseline_ms / best_ms
        if isinstance(baseline_ms, (int, float))
        and isinstance(best_ms, (int, float))
        and best_ms > 0
        else None
    )
    accepted_rows = [row for row in TRAJECTORY if row.get("accepted")]
    rejected_rows = [row for row in TRAJECTORY if not row.get("accepted")]

    block_match = re.search(r"block_size\s*=\s*(\d+)", best_source)
    warps_match = re.search(r"num_warps\s*=\s*(\d+)", best_source)
    best_config = {
        "block_size": int(block_match.group(1)) if block_match else None,
        "num_warps": int(warps_match.group(1)) if warps_match else None,
    }

    SUMMARY = {
        "workspace": str(WORKSPACE_DIR),
        "model": os.environ["KERNEL_AGENT_MODEL"],
        "gpu": GPU_NAME,
        "gpu_arch": GPU_ARCH or None,
        "rocm": torch.version.hip,
        "rounds": len(TRAJECTORY),
        "accepted": len(accepted_rows),
        "rejected": len(rejected_rows),
        "baseline_median_ms": baseline_ms,
        "best_median_ms": best_ms,
        "speedup": speedup,
        "best_config": best_config,
        "bound": BEST_PROFILE.get("bound"),
        "bottleneck": BEST_PROFILE.get("bottleneck"),
    }
    (WORKSPACE_DIR / "run_summary.json").write_text(
        json.dumps(SUMMARY, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

    failed_lines = []
    for index, row in enumerate(TRAJECTORY, 1):
        if row.get("accepted"):
            continue
        failed_lines.append(
            f"- R{index}: {row.get('change') or '(未记录改动)'}；"
            f"reason={row.get('reason') or row.get('status')}"
        )
    if not failed_lines:
        failed_lines.append("- 本次轨迹没有被拒绝的候选。")

    speedup_text = f"{speedup:.2f}×" if speedup is not None else "不可计算"
    import textwrap

    report = textwrap.dedent(f"""\
    # vector_add 多轮优化报告

    ## 环境与任务
    - Model: {os.environ['KERNEL_AGENT_MODEL']}
    - GPU: {GPU_NAME} ({GPU_ARCH or 'unknown'})
    - ROCm / PyTorch / Triton: {torch.version.hip} / {torch.__version__} / {triton.__version__}
    - Shape / dtype: 4096×2048 / fp16
    - 晋升阈值: {threshold:.0%}

    ## 结果
    - 评测轮次: {len(TRAJECTORY)}
    - 接受 / 拒绝: {len(accepted_rows)} / {len(rejected_rows)}
    - Baseline median: {baseline_ms} ms
    - Best median: {best_ms} ms
    - Baseline → Best: {speedup_text}
    - Best 配置: block_size={best_config['block_size']}, num_warps={best_config['num_warps']}
    - Bound / bottleneck: {BEST_PROFILE.get('bound')} / {BEST_PROFILE.get('bottleneck')}

    ## 失败与回退
    {chr(10).join(failed_lines)}

    ## 证据
    - trajectory: {WORKSPACE_DIR / 'trajectory.jsonl'}
    - baseline benchmark: {WORKSPACE_DIR / 'baseline-benchmark.json'}
    - best benchmark: {WORKSPACE_DIR / 'best-benchmark.json'}
    - best profile: {WORKSPACE_DIR / 'best-profile.json'}
    - visualizations: {WORKSPACE_DIR / 'viz'}
    - agent status: {WORKSPACE_DIR / 'agent-status.json'}
    """)
    (WORKSPACE_DIR / "comparison-report.md").write_text(report, encoding="utf-8")
    print(report)


## Expected Output / Interpretation

依赖单元应打印 `Matplotlib 中文字体: ...`。后续生成三张图片时不应再出现 `Glyph ... missing from font(s) DejaVu Sans` 警告。

1. baseline 和 best 都通过正确性检查，并分别留下 benchmark JSON；
2. `trajectory.jsonl` 中包含每轮 `change/status/accepted/latencyMs/improvementFraction`；
3. 被拒绝的候选保留在轨迹中，但不会覆盖 `best.py`；
4. `viz/` 下生成 `rounds_overview.png`、`status_breakdown.png` 和 `process_timeline.png`；
5. `comparison-report.md` 中的数字与当前工作区证据一致。

若出现 `429 / process_concurrency_rate_limit_exceeded`，Notebook 会自动限速并退避重试。重试耗尽时保留当前工作区；等待平台恢复后，只需重新运行 Agent 步骤，不必重跑已完成的 GPU 工具。

`trajectory.jsonl` 只由 `accept_candidate` 创建。若没有任何候选完成 benchmark，Agent 会明确报告“未生成轨迹”，不会再输出一个不存在的文件路径。更新 `code/` 后若当前 kernel 仍缓存旧模块，请重启 kernel，再从模型配置步骤运行到步骤5。

当最大步数恰好停在新候选的 compile 阶段时，该未完成候选不会写入轨迹，也不会影响前面已完成的接受/拒绝记录；后续分析只使用已有权威轨迹。


## Pass Criteria

1. 完成一次多轮 Agent 搜索并产生非空轨迹；
2. 能指出至少一轮接受或拒绝的权威依据；
3. 能从曲线解释最大收益、回退和后期收益变化；
4. 能区分正文中的历史参考结果与当前云端结果；
5. 对比报告同时记录成功候选和失败候选。


## 本章小结

- 实战首先证明闭环可信，再追求 3–5×；算子选型决定加速叙事。
- 本轮真实结果：10 评 / 5 接受，**0.705→0.322 ms（≈2.19×）**，有效机制是更大 block + 更高 warps。
- 失败回退由「不更新 best」保证；轨迹 + 可视化是报告账本。
- Agent 擅长执行与留痕；关键结构洞察仍常需人机协作。



## 延伸阅读

- [算子优化 Agent 实战报告 · vector_add](../../docs/part3-agent/chapter17/optimization-report.md)
- `code/part3-agent/chapter14/EXPERIMENT.md`
